# LC + controlled redundancy -> route+conn+adj edit model (norm-Huber critic)

Консолидированный ноутбук (заменяет `train_conn_adj_mixed` + `critic_normalization_experiment`).

**Что делает (Run-All):**
1. Генерирует **LC-датасет (1000 графов, 50 узлов, контракт 12 маршрутов / len 8–15)**.
   LC строится с **крайними weight-combo** (demand 1/0/0, route 0/1/0, conn 0/0/1) — по 33%
   для разнообразия. В маршруты вносится **контролируемая избыточность** (over-extend концов +
   out-and-back дублирование), стратифицировано **clean / low / mid** по 33%.
2. Обучает edit-модель на **route + connectivity + adj** (demand off), adj — **фикс** cap
   (target=0.15, W=10). **Cost-веса route/conn варьируются при обучении** (агент кондишенится
   на них), eval — на **граничных** весах.
3. Критик — **norm + Huber + value-clip** (как в прошлом critic-эксперименте).
4. Выводит **все метрики критика**, **кривые актора**, **redundancy до/после** и **разбивку по K**.

In [ ]:
import os, sys, pickle, shutil, random as _random
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from hydra import compose, initialize_config_dir
from IPython.display import display

from eval_lib.context import (ROOT_DIR, CFG_DIR, DATASETS_DIR,
                              MODEL_OUTPUTS_DIR, EDIT_MODEL_WEIGHTS_DIR)
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
from connectpt.routes_generator import utils as lrnu
from connectpt.routes_generator.improvement_learning import (
    load_raw_graphs_and_lc_routes, make_improvement_batch,
    rollout_lc_improvement, train_lc_improvement_cfg)
from connectpt.routes_generator.torch_utils import (
    get_batch_tensor_from_routes, dump_routes)
from connectpt.routes_generator.transit_time_estimator import RouteGenBatchState
from connectpt.routes_generator.citygraph_dataset import (
    STOP_KEY, DynamicCityGraphDataset)
from connectpt.routes_generator.bee_colony import get_adjustment_degrees
from torch_geometric.data import Batch
from eval_lib.results_io import save_table
from eval_lib import build_lc_cfg, run_lc, as_route_tensor

pd.set_option("display.max_columns", None)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Конфигурация

In [ ]:
# --- датасет ---
N_GRAPHS        = 1000
RAW_N_NODES     = 50
RAW_GRAPH_TYPE  = "mixed"
RAW_GRAPH_SEED  = 0
TARGET_N_ROUTES = 12
MIN_ROUTE_LEN   = 8
MAX_ROUTE_LEN   = 15
LC_N_SAMPLES    = 1
# LC weight-combo (крайние), по 33%:  (demand, route, conn, tag)
LC_COMBOS = [(1.0, 0.0, 0.0, "demand"), (0.0, 1.0, 0.0, "route"), (0.0, 0.0, 1.0, "conn")]
# уровни избыточности (число инъекций K), по 33%
REDUN_LEVELS = {"clean": 0, "low": 2, "mid": 4}
DATASET_DIRNAME = "lc_redundant_n50_r12_len8_15"
NEW_DATASET_DIR = DATASETS_DIR / DATASET_DIRNAME
SUBSET_PKL      = NEW_DATASET_DIR / "raw_graphs_subset.pkl"
META_CSV        = NEW_DATASET_DIR / "meta.csv"
FORCE_REGEN     = False

# --- objective route+conn+adj ---
DISABLED_COST_COMPONENTS = ["demand"]      # активны route + connectivity
ADJ_OBJECTIVE = "cap"
ADJ_WEIGHT    = 10.0
ADJ_TARGET    = 0.15
ADJ_MODE      = "paper"
ADJ_GAP       = 0.1
# cost-веса route/conn варьируются при обучении (агент кондишенится):
VARY_WEIGHTS  = True
OP_FRACTION   = 0.4    # доля «route-only» сэмплов
MCW_FRACTION  = 0.4    # доля «conn-only» сэмплов (остальное — intermediate route/conn)

# --- критик: norm + Huber + value-clip ---
CRITIC_OVERRIDES = ["++critic_normalize_returns=true", "++critic_huber=true",
                    "++critic_huber_delta=1.0", "++critic_value_clip=0.2"]

# --- обучение ---
N_ITERATIONS  = 300
BATCH_SIZE    = 16
TRAIN_FRACTION = 0.9
SPLIT_SEED    = 0
MAX_ROUTE_EDIT_STEPS = MAX_ROUTE_LEN
MAX_TRIM_ACTIONS_PER_ROUTE = 1

# --- eval: граничные cost-веса (route, conn); demand=0 ---
EVAL_WEIGHT_COMBOS = [(1.0, 0.0, "route-only"), (0.0, 1.0, "conn-only"),
                      (0.5, 0.5, "balanced")]
EVAL_N_PER_LEVEL = 12      # графов val на каждый уровень K при оценке

RUN_NAME = "lc_redundant_route_conn_adj"
print(f"{N_GRAPHS} graphs, contract {TARGET_N_ROUTES}r/len{MIN_ROUTE_LEN}-{MAX_ROUTE_LEN}")
print(f"objective: route+conn+adj (disabled={DISABLED_COST_COMPONENTS}), adj cap t={ADJ_TARGET} W={ADJ_WEIGHT}")
print(f"vary cost-weights={VARY_WEIGHTS}; critic={CRITIC_OVERRIDES}; n_iter={N_ITERATIONS}")

## Генерация LC + контролируемая избыточность (на диск)

LC-маршруты (крайний weight-combo) -> приведение к контракту -> инъекции избыточности
(`over-extend` концов и `out-and-back` дублирование, обе удаляемы trim'ом, в пределах max_len).
Сохраняется в формате `load_raw_graphs_and_lc_routes` + `meta.csv` (combo, level, K, redund%).

In [ ]:
def _to_fixed(routes):
    t = as_route_tensor(routes).long()
    if t.ndim == 3:
        t = t[0]
    if t.shape[0] < TARGET_N_ROUTES:
        t = torch.cat([t, torch.full((TARGET_N_ROUTES - t.shape[0], t.shape[1]), -1, dtype=t.dtype)], 0)
    else:
        t = t[:TARGET_N_ROUTES]
    if t.shape[1] < MAX_ROUTE_LEN:
        t = torch.cat([t, torch.full((t.shape[0], MAX_ROUTE_LEN - t.shape[1]), -1, dtype=t.dtype)], 1)
    elif t.shape[1] > MAX_ROUTE_LEN:
        t = t[:, :MAX_ROUTE_LEN]
    return t


def _tensors(g):
    return {"node_locs": g[STOP_KEY].pos.detach().cpu().clone(),
            "street_adj": g.street_adj.detach().cpu().clone(),
            "demand": g.demand.detach().cpu().clone()}


def _redundancy(routes):
    """(edges-uniq)/edges по undirected мультимножеству рёбер всех маршрутов."""
    cnt = Counter()
    for r in routes.tolist():
        r = [x for x in r if x >= 0]
        for a, b in zip(r[:-1], r[1:]):
            cnt[(min(a, b), max(a, b))] += 1
    tot = sum(cnt.values())
    return 0.0 if tot == 0 else (tot - len(cnt)) / tot


def inject_redundancy(routes, street_adj, n_inject, rng):
    """over-extend / out-and-back добавления к концам маршрутов (в пределах max_len).
    Создают удаляемую trim'ом избыточность, сохраняя валидность пути."""
    adj = torch.isfinite(street_adj) & (street_adj > 0)
    routes = routes.clone()
    R = routes.shape[0]
    for _ in range(n_inject):
        lens = (routes >= 0).sum(1)
        cand = [r for r in range(R) if 2 <= int(lens[r]) <= MAX_ROUTE_LEN - 2]
        if not cand:
            break
        r = rng.choice(cand); l = int(lens[r])
        last = int(routes[r, l - 1])
        nbrs = [n for n in torch.where(adj[last])[0].tolist() if n != last]
        if not nbrs:
            continue
        y = rng.choice(nbrs)
        # out-and-back: last -> y -> last  (ребро last-y покрыто дважды; trim снимет)
        routes[r, l] = y
        routes[r, l + 1] = last
    return routes


def generate_dataset():
    if NEW_DATASET_DIR.exists():
        shutil.rmtree(NEW_DATASET_DIR)
    NEW_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    _random.seed(RAW_GRAPH_SEED); torch.manual_seed(RAW_GRAPH_SEED)
    ds = DynamicCityGraphDataset(min_nodes=RAW_N_NODES, max_nodes=RAW_N_NODES,
                                 data_type=RAW_GRAPH_TYPE, mumford_style=True, pos_only=False)
    raw = [ds.generate_graph(n_nodes=RAW_N_NODES) for _ in range(N_GRAPHS)]
    level_names = list(REDUN_LEVELS.keys())
    subset, meta = [], []
    for gi, g in enumerate(raw):
        combo = LC_COMBOS[gi % len(LC_COMBOS)]
        level = level_names[(gi // 1) % len(level_names)] if False else level_names[gi % len(level_names)]
        K = REDUN_LEVELS[level]
        d, rt, cn, ctag = combo
        c = build_lc_cfg(run_name=f"lcred_{gi}", n_routes=TARGET_N_ROUTES,
                         min_route_len=MIN_ROUTE_LEN, max_route_len=MAX_ROUTE_LEN,
                         demand_time_weight=d, route_time_weight=rt,
                         median_connectivity_weight=cn)
        _, _, _, r, _ = run_lc(c, tensors=_tensors(g), run_name_prefix="lcred_",
                               n_samples=LC_N_SAMPLES)
        routes = _to_fixed(r)
        rb = _redundancy(routes)
        rng = _random.Random(1000 + gi)
        routes = inject_redundancy(routes, g.street_adj, K, rng)
        ra = _redundancy(routes)
        gdir = NEW_DATASET_DIR / f"graph_{gi:04d}"; gdir.mkdir(parents=True, exist_ok=True)
        dump_routes(f"lc_lcred_graph_{gi:04d}_routes", routes, out_dir=gdir)
        subset.append(g)
        meta.append({"graph_index": gi, "lc_combo": ctag, "redun_level": level,
                     "K": K, "redun_before_lc": round(rb, 4), "redun_after_inject": round(ra, 4)})
        if (gi + 1) % 100 == 0:
            print(f"  {gi+1}/{N_GRAPHS}")
    with SUBSET_PKL.open("wb") as fh:
        pickle.dump(subset, fh)
    pd.DataFrame(meta).to_csv(META_CSV, index=False)
    print(f"Saved {len(subset)} graphs -> {NEW_DATASET_DIR}")


_have = len(list(NEW_DATASET_DIR.glob("graph_*"))) if NEW_DATASET_DIR.exists() else 0
if SUBSET_PKL.exists() and _have == N_GRAPHS and META_CSV.exists() and not FORCE_REGEN:
    print(f"Датасет уже на диске ({_have} graphs) -> пропуск генерации")
else:
    if _have and _have != N_GRAPHS:
        print(f"На диске {_have}, нужно {N_GRAPHS} -> перегенерация")
    generate_dataset()

## Чтение датасета + meta + split

In [ ]:
graphs, seed_routes = load_raw_graphs_and_lc_routes(SUBSET_PKL, NEW_DATASET_DIR)
meta_df = pd.read_csv(META_CSV)
N = len(graphs)
print(f"loaded {N} graphs; seed_routes={tuple(seed_routes.shape)}")
print("redundancy after injection by level:")
display(meta_df.groupby("redun_level")[["K", "redun_before_lc", "redun_after_inject"]].mean().round(3))

_perm = torch.randperm(N, generator=torch.Generator().manual_seed(SPLIT_SEED))
_ntr = int(TRAIN_FRACTION * N)
TRAIN_INDICES = _perm[:_ntr].clone()
VAL_INDICES = _perm[_ntr:].clone()
LEVEL_OF = dict(zip(meta_df["graph_index"], meta_df["redun_level"]))
print(f"train={len(TRAIN_INDICES)} val={len(VAL_INDICES)}")

## Модель (route+conn+adj, фикс adj) + варьируемые cost-веса + norm-Huber критик

In [ ]:
overrides = [
    "model=bestsofar_feb2023_trim",
    "model.route_generator.kwargs.serial_halting=True",
    f"++run_name={RUN_NAME}",
    "++experiment.logdir=null",
    f"++adjustment_degree_weight={float(ADJ_WEIGHT)}",
    f"++adjustment_degree_target={float(ADJ_TARGET)}",
    f"++adjustment_degree_objective={ADJ_OBJECTIVE}",
    f"++adjustment_degree_gap={float(ADJ_GAP)}",
    f"++adjustment_degree_mode={ADJ_MODE}",
] + CRITIC_OVERRIDES
with initialize_config_dir(config_dir=str(CFG_DIR), version_base=None):
    cfg = compose(config_name="ppo_50nodes.yaml", overrides=overrides)
device, run_name, _, cost_obj, model = lrnu.process_standard_experiment_cfg(
    cfg, run_name_prefix="improvement_")
cost_obj.ignore_stops_oob = True
cost_obj.set_enabled_components(disabled_components=DISABLED_COST_COMPONENTS or None)
# Варьировать cost-веса route/conn при обучении (агент кондишенится на них).
if VARY_WEIGHTS:
    cost_obj.variable_weights = True
    cost_obj.pp_fraction = 0.0          # demand отключён -> 0
    cost_obj.op_fraction = OP_FRACTION  # route-only доля
    cost_obj.mcw_fraction = MCW_FRACTION  # conn-only доля
BEST_MODEL_PATH = EDIT_MODEL_WEIGHTS_DIR / f"{run_name}.pt"
print(f"run_name={run_name} | enabled={list(cost_obj.enabled_component_names)} | "
      f"variable_weights={cost_obj.variable_weights}")
print(f"critic: normalize={cfg.get('critic_normalize_returns')} huber={cfg.get('critic_huber')} "
      f"value_clip={cfg.get('critic_value_clip')}")
print(f"best -> {BEST_MODEL_PATH}")

## Обучение (300 эпох)

In [ ]:
train_result = train_lc_improvement_cfg(
    model=model, cost_obj=cost_obj, graphs=graphs, seed_routes=seed_routes,
    device=device, cfg=cfg, output_dir=MODEL_OUTPUTS_DIR, run_name=run_name,
    train_fraction=TRAIN_FRACTION, batch_size=BATCH_SIZE,
    min_route_len=MIN_ROUTE_LEN, max_route_len=MAX_ROUTE_LEN, seed=SPLIT_SEED,
    max_route_edit_steps=MAX_ROUTE_EDIT_STEPS,
    max_trim_actions_per_route=MAX_TRIM_ACTIONS_PER_ROUTE,
    target_n_routes=TARGET_N_ROUTES,
    train_indices=TRAIN_INDICES, val_indices=VAL_INDICES,
    best_model_path=BEST_MODEL_PATH, n_iterations=N_ITERATIONS,
)
history_df = pd.DataFrame(train_result["history"])
save_table(history_df, f"{RUN_NAME}_training_history")
print(f"history rows={len(history_df)}; best -> {BEST_MODEL_PATH}")

## Кривые обучения актора

In [ ]:
h = history_df
def _num(col):
    return pd.to_numeric(h[col], errors="coerce") if col in h.columns else None

fig, ax = plt.subplots(2, 3, figsize=(16, 8), constrained_layout=True)
panels = [("train_reward_mean", "train reward"), ("val_delta", "val cost delta (+=улучш.)"),
          ("val_win_rate", "val win rate"), ("val_component_delta_route", "val route delta"),
          ("val_component_delta_connectivity", "val conn delta"),
          ("val_changed_route_rate", "changed-route rate")]
for a, (col, title) in zip(ax.flat, panels):
    y = _num(col)
    if y is not None and y.notna().any():
        a.plot(h["epoch"], y, marker="o", ms=2)
    a.axhline(0, color="k", lw=0.8); a.set_title(title); a.set_xlabel("epoch"); a.grid(alpha=0.25)
fig.suptitle("Actor training curves", fontsize=13, fontweight="bold")
plt.show(); plt.close(fig)

## Метрики критика (учится ли)

Все доступные critic-колонки. `explained_variance` — главное (scale-free): растёт к 1 = критик
хорошо предсказывает returns. `critic_mse` — в нормированном Huber-пространстве (тренд важен,
не абсолют).

In [ ]:
crit_cols = [c for c in h.columns if "critic" in c.lower()]
print("critic columns:", crit_cols)
if crit_cols:
    n = len(crit_cols)
    fig, ax = plt.subplots(1, n, figsize=(5 * n, 4), squeeze=False, constrained_layout=True)
    for a, col in zip(ax[0], crit_cols):
        y = _num(col)
        if y is not None and y.notna().any():
            a.plot(h["epoch"], y, marker="o", ms=2, color="tab:orange")
        a.set_title(col, fontsize=9); a.set_xlabel("epoch"); a.grid(alpha=0.25)
        if "explained" in col:
            a.axhline(0, color="k", lw=0.8)
    fig.suptitle("Critic metrics", fontsize=13, fontweight="bold")
    plt.show(); plt.close(fig)
    display(h[["epoch"] + crit_cols].tail(8).round(4))

## Оценка: redundancy до/после + разбивка по уровням K + граничные веса

Модель прогоняется на val-подмножестве при **граничных** cost-весах (route-only / conn-only /
balanced). Для каждого уровня K (clean/low/mid) меряем: redundancy seed vs после агента,
route/conn-вклад, adj. Главное — **падает ли redundancy** (агент убирает дубли) на low/mid.

In [ ]:
def _redun_t(routes_2d):
    cnt = Counter()
    for r in routes_2d.tolist():
        r = [x for x in r if x >= 0]
        for a, b in zip(r[:-1], r[1:]):
            cnt[(min(a, b), max(a, b))] += 1
    tot = sum(cnt.values())
    return 0.0 if tot == 0 else (tot - len(cnt)) / tot

# val-подмножество по уровням
val_by_level = {lv: [] for lv in REDUN_LEVELS}
for gi in VAL_INDICES.tolist():
    lv = LEVEL_OF.get(gi)
    if lv in val_by_level and len(val_by_level[lv]) < EVAL_N_PER_LEVEL:
        val_by_level[lv].append(gi)

base_w = cost_obj.get_weights(device)
def mkw(rw, cw):
    w = {k: (v.clone() if torch.is_tensor(v) else v) for k, v in base_w.items()}
    w["demand_time_weight"] = torch.as_tensor(0.0, device=device)
    w["route_time_weight"] = torch.as_tensor(float(rw), device=device)
    w["median_connectivity_weight"] = torch.as_tensor(float(cw), device=device)
    return w

rows = []
model.eval()
for rw, cw, wtag in EVAL_WEIGHT_COMBOS:
    weights = mkw(rw, cw)
    for lv, idxs in val_by_level.items():
        if not idxs:
            continue
        rb_b, rb_a, adjs, rcomp, ccomp = [], [], [], [], []
        for gi in idxs:
            gb, rb = make_improvement_batch(graphs, seed_routes, torch.tensor([gi]),
                                            device, training=False, target_n_routes=TARGET_N_ROUTES)
            with torch.no_grad():
                out = rollout_lc_improvement(model, cost_obj, gb, rb, MIN_ROUTE_LEN, MAX_ROUTE_LEN,
                                             greedy=True, cost_weights=weights,
                                             max_route_edit_steps=MAX_ROUTE_EDIT_STEPS,
                                             max_trim_actions_per_route=MAX_TRIM_ACTIONS_PER_ROUTE)
            st = out[0]; res = cost_obj(st)
            comps, wts, _ = cost_obj._compute_cost_components_from_result(st, res)
            comps = comps.reshape(-1, 3); wts = wts.reshape(-1, 3)
            rcomp.append((comps[:, 1] * wts[:, 1]).item()); ccomp.append((comps[:, 2] * wts[:, 2]).item())
            imp = get_batch_tensor_from_routes(st.routes, device, max_route_len=rb.shape[-1])
            rb_b.append(_redun_t(rb[0])); rb_a.append(_redun_t(imp[0]))
            nr = min(imp.shape[1], rb.shape[1]); ll = min(imp.shape[-1], rb.shape[-1])
            adjs.append(get_adjustment_degrees(imp[:, :nr, :ll], rb[:, :nr, :ll],
                        cost_obj.symmetric_routes, gap=ADJ_GAP, mode=ADJ_MODE).mean().item())
        rows.append({"eval_weights": wtag, "K_level": lv, "n": len(idxs),
                     "redun_seed": round(float(np.mean(rb_b)), 3),
                     "redun_after": round(float(np.mean(rb_a)), 3),
                     "redun_drop": round(float(np.mean(rb_b) - np.mean(rb_a)), 3),
                     "route_w": round(float(np.mean(rcomp)), 3),
                     "conn_w": round(float(np.mean(ccomp)), 3),
                     "adj_vs_seed": round(float(np.mean(adjs)), 3)})
eval_df = pd.DataFrame(rows)
display(eval_df)
save_table(eval_df, f"{RUN_NAME}_eval_by_level_weights")
print("redun_drop>0 => агент убирает избыточность; смотри особенно low/mid уровни.")